# L2 Memory Modern

这个版本保留原始 `L2-Memory.ipynb` 的学习目标，但把已经过时的组件换成了 LangChain 1.x 推荐写法：

- `ConversationChain` -> `ChatPromptTemplate + RunnableWithMessageHistory`
- `ConversationBufferMemory` -> `InMemoryChatMessageHistory`
- `ConversationBufferWindowMemory` -> `trim_messages`
- `ConversationTokenBufferMemory` -> 自定义 token 统计 + `trim_messages`
- `ConversationSummaryBufferMemory` -> “摘要链 + 最近对话” 的组合

重点不是机械替换类名，而是理解新版里“记忆”已经更像一组可组合组件：
`消息历史对象 + Prompt + Runnable 包装器 + 可选裁剪/摘要逻辑`


In [1]:
import os
from collections import defaultdict
from dotenv import load_dotenv, find_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage, trim_messages
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

_ = load_dotenv(find_dotenv())

# 这里继续使用你课程环境里统一的 qwen-max。
# ChatOpenAI 只是 OpenAI 兼容客户端，不代表底层一定是 OpenAI 模型。
llm = ChatOpenAI(
    temperature=0.0,
    model="qwen-max",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
)


In [4]:
# 这就是新版最核心的“记忆仓库”。
# 旧版 Memory 类经常把“存储、裁剪、拼 Prompt”绑在一起；
# 新版更推荐你自己显式管理历史记录，这样更灵活。
history_store: dict[str, InMemoryChatMessageHistory] = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    # 如果当前 session 还没有历史，就创建一个新的内存历史对象。
    if session_id not in history_store:
        history_store[session_id] = InMemoryChatMessageHistory()
    return history_store[session_id]

# Prompt 里显式放一个 history 占位符，表示“这里要插入历史消息”。
# 这一步取代了旧版 ConversationChain 自动拼接 memory 的做法。
memory_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        MessagesPlaceholder(variable_name="history"), # 处理对话历史
        ("human", "{input}"),
    ]
)

# 先定义“纯对话链”。
# 它本身并不记忆任何内容，只负责：Prompt -> LLM -> 字符串输出。
base_conversation_chain = memory_prompt | llm | StrOutputParser()

# 再把“纯链”包上一层带历史的 Runnable。
# 这样每次 invoke 时，LangChain 会自动：
# 1. 读取当前 session 的历史
# 2. 填到 Prompt 的 history 位置
# 3. 调用模型
# 4. 把本轮 user / assistant 消息写回历史
# 实例化一个支持消息历史记录的可运行对象（Runnable）
conversation = RunnableWithMessageHistory(
    # 第一个参数：基础对话链。这是你的核心逻辑（通常包含 Prompt + LLM + OutputParser）
    base_conversation_chain,
    # 获取会话历史的函数。该函数通常接收一个 session_id 并返回对应的 BaseChatMessageHistory 对象
    # 它决定了聊天记录是存放在内存、数据库（如 Redis）还是本地文件中
    get_session_history=get_session_history,
    # 输入消息的键名。告诉包装器，在调用时输入字典里的哪个键（Key）代表当前用户的最新提问
    input_messages_key="input",
    # 历史消息的键名。对应你 PromptTemplate 中用于存放历史对话占位符的变量名
    # 包装器会自动从 get_session_history 获取记录并注入到这个键中
    history_messages_key="history",
)


In [5]:
# 新版里“按会话记忆”通常通过 config 里的 session_id 实现。
session_config = {"configurable": {"session_id": "demo-user"}}

print(conversation.invoke({"input": "Hi, my name is Andrew."}, config=session_config))
print(conversation.invoke({"input": "What is 1 + 1?"}, config=session_config))
print(conversation.invoke({"input": "What is my name?"}, config=session_config))


Hello Andrew! It's nice to meet you. How can I assist you today?
1 + 1 equals 2. Is there anything else you'd like to know or discuss?
Your name is Andrew. Is there anything else you'd like to talk about or any other questions you have?


In [6]:
# 直接查看当前 session 的底层消息对象。
# 这比旧版 memory.buffer 更原始，但也更清楚：你能看到每条消息的角色和内容。
demo_history = get_session_history("demo-user")

for index, message in enumerate(demo_history.messages, start=1):
    print(f"{index}. {message.type}: {message.content}")


1. human: Hi, my name is Andrew.
2. ai: Hello Andrew! It's nice to meet you. How can I assist you today?
3. human: What is 1 + 1?
4. ai: 1 + 1 equals 2. Is there anything else you'd like to know or discuss?
5. human: What is my name?
6. ai: Your name is Andrew. Is there anything else you'd like to talk about or any other questions you have?


In [7]:
# 下面演示“手动写入 / 手动读取历史”。
# 这对应旧版 save_context / load_memory_variables 的学习目标。
manual_history = InMemoryChatMessageHistory()

# 手动追加用户和助手消息。
manual_history.add_user_message("Hi")
manual_history.add_ai_message("What's up?")
manual_history.add_user_message("Not much, just hanging")
manual_history.add_ai_message("Cool")

print("手动写入后的历史：")
for message in manual_history.messages:
    print(f"- {message.type}: {message.content}")


手动写入后的历史：
- human: Hi
- ai: What's up?
- human: Not much, just hanging
- ai: Cool


In [10]:
# Window Memory 的核心思想不是“专门一个类”，而是：
# 需要时只截取最近几条消息送给模型。
#
# 这里用 trim_messages 保留“最近 4 条消息”。
# 因为一轮对话通常是 Human + AI 两条消息，所以 4 条大致等于最近 2 轮。
recent_messages = trim_messages(
    manual_history.messages,
    max_tokens=4,
    token_counter=len,   # 这里故意把“每条消息算 1 个 token”，方便教学演示窗口效果。
    strategy="last",
)

print("Window Memory 效果（只保留最近几条消息）：")
for message in recent_messages:
    print(f"- {message.type}: {message.content}")


Window Memory 效果（只保留最近几条消息）：
- human: Hi
- ai: What's up?
- human: Not much, just hanging
- ai: Cool


In [11]:
# Token Buffer Memory 的思想和 Window Memory 很像，只是裁剪标准从“消息条数”改成“token 预算”。
# 为了不依赖特定模型分词器，这里用一个近似计数函数：按单词数粗略估算 token。
def approximate_token_count(messages):
    total = 0
    for message in messages:
        total += len(str(message.content).split())
    return total

token_history = InMemoryChatMessageHistory()
token_history.add_user_message("AI is what?")
token_history.add_ai_message("Amazing!")
token_history.add_user_message("Backpropagation is what?")
token_history.add_ai_message("Beautiful!")
token_history.add_user_message("Chatbots are what?")
token_history.add_ai_message("Powerful!")

token_limited_messages = trim_messages(
    token_history.messages,
    max_tokens=8,
    token_counter=approximate_token_count,
    strategy="last",
)

print("Token Buffer 效果（超出预算时只保留最近部分）：")
for message in token_limited_messages:
    print(f"- {message.type}: {message.content}")


Token Buffer 效果（超出预算时只保留最近部分）：
- human: Backpropagation is what?
- ai: Beautiful!
- human: Chatbots are what?
- ai: Powerful!


In [13]:
# Summary Buffer Memory 的核心不是某个类，而是一个模式：
# “把很久之前的对话压缩成摘要，再加上最近若干条原始消息”。
#
# 下面我们手动搭一个最小实现，便于看清楚流程。
# --- 1. 定义原始数据 ---
# 这是一个包含全天安排的字符串，作为我们要处理的初始背景信息
schedule = (
    "There is a meeting at 8am with your product team. "
    "You need your presentation prepared. "
    "From 9am to 12pm you can work on your LangChain project. "
    "At noon you have lunch with a customer. "
    "At 3pm you meet the marketing team. "
    "At 5pm you review customer feedback."
)

# --- 2. 构建摘要提示模板 ---
# 使用 from_messages 创建多角色模板，明确区分系统指令和用户输入
summary_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你负责维护对话的滚动摘要。请更新现有摘要，确保其仍然捕获了所有重要的事实。"
        ),
        (
            "human",
            "现有摘要：\n{existing_summary}\n\n"
            "新的对话轮次：\n{new_turns}"
        ),
    ]
)

# --- 3. 构建 LCEL 链 ---
# 使用管道符 | 将模板、大模型实例和字符串解析器连接起来
summary_chain = summary_prompt | llm | StrOutputParser()

# --- 4. 初始化存储 ---
# 实例化一个内存消息历史对象，用于存放原始的对话信息
summary_history = InMemoryChatMessageHistory()
# 初始化一个空字符串，用于存放不断更新的摘要结果
running_summary = ""

# --- 5. 定义摘要逻辑函数 ---
def summarize_history(history: InMemoryChatMessageHistory, existing_summary: str) -> str:
    # 步骤 A：将历史对象中最近的这批消息格式化为纯文本字符串
    turns_text = "\n".join(
        f"{message.type}: {message.content}"
        for message in history.messages
    )
    # 步骤 B：调用摘要链，将旧摘要和新消息喂给模型，生成合并后的新摘要
    return summary_chain.invoke(
        {
            "existing_summary": existing_summary or "No summary yet.",
            "new_turns": turns_text,
        }
    )

# --- 6. 执行流程演示 ---
# 向历史记录中添加用户的日程信息
summary_history.add_user_message(f"Here is my schedule: {schedule}")
# 向历史记录中添加 AI 的确认回复
summary_history.add_ai_message("Thanks, I noted your schedule.")

# 触发函数：生成第一版的滚动摘要
running_summary = summarize_history(summary_history, running_summary)

# 打印输出结果
print("第一次摘要：")
print(running_summary)

第一次摘要：
摘要：
You have a meeting at 8am with the product team, for which you need to prepare a presentation. From 9am to 12pm, time is allocated for working on your LangChain project. At noon, there's a lunch appointment with a customer. A meeting with the marketing team is set for 3pm, and at 5pm, you are scheduled to review customer feedback.


In [14]:
# 继续追加一轮新对话，然后再次更新摘要。
summary_history.add_user_message("What would be a good demo to show the product team?")
summary_history.add_ai_message(
    "A concise demo showing memory, retrieval, and tool use would fit that audience well."
)

running_summary = summarize_history(summary_history, running_summary)

print("更新后的摘要：")
print(running_summary)

# 在生产里，常见做法是：
# 1. 保留 running_summary
# 2. 只保留最近几条原始消息
# 3. 后续 Prompt 同时注入“摘要 + 最近消息”
recent_for_prompt = trim_messages(
    summary_history.messages,
    max_tokens=4,
    token_counter=len,
    strategy="last",
)

print("\n最近原始消息：")
for message in recent_for_prompt:
    print(f"- {message.type}: {message.content}")


更新后的摘要：
更新后的摘要：
你有一个与产品团队的会议在早上8点，为此你需要准备一个演示文稿。从上午9点到中午12点的时间被分配用于你的LangChain项目工作。中午时分，你安排了与一位客户的午餐约会。下午3点，你将与市场团队会面；而在下午5点，你计划审阅客户反馈。对于产品团队来说，一个展示记忆、检索和工具使用功能的简短演示将是合适的。

最近原始消息：
- human: Here is my schedule: There is a meeting at 8am with your product team. You need your presentation prepared. From 9am to 12pm you can work on your LangChain project. At noon you have lunch with a customer. At 3pm you meet the marketing team. At 5pm you review customer feedback.
- ai: Thanks, I noted your schedule.
- human: What would be a good demo to show the product team?
- ai: A concise demo showing memory, retrieval, and tool use would fit that audience well.
